# 07 — Focused tuning and one-time outer evaluation

This notebook tunes only the two pipeline/model combinations retained for each split seed and target. It then refits one selected direct, Stage 1, and Stage 2 pipeline on each full outer-training set and evaluates the untouched outer-test patients once.

**Selection boundary:** all tuning and winner selection use the five inner folds. Outer-test results are saved for reporting and cannot change any pipeline, hyperparameter, model family, or sensitivity decision.

## 1. Check the modeling environment

Confirm that the three external model packages used by the frozen nine-family scope are available. The CPU setting prevents a noisy joblib hardware-detection warning.

In [1]:
import importlib.util
import os

# Avoid noisy macOS physical-core detection in joblib while preserving the
# available logical-core limit for parallel estimators.
os.environ.setdefault(
    "LOKY_MAX_CPU_COUNT",
    str(max(1, (os.cpu_count() or 2) - 1)),
)

required_external_packages = ["xgboost", "lightgbm", "catboost"]
missing_packages = [
    package
    for package in required_external_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Install the missing packages in this notebook environment, restart the "
        f"kernel, and rerun: {missing_packages}"
    )
print("External model packages are available.")

External model packages are available.


## 2. Import the analysis tools

Load the data, statistical, preprocessing, modeling, and evaluation tools used below.

In [2]:
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn import __version__ as sklearn_version
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, recall_score, roc_auc_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier, __version__ as xgboost_version
from lightgbm import LGBMClassifier, __version__ as lightgbm_version
from catboost import CatBoostClassifier, __version__ as catboost_version

## 3. Locate the final-run inputs

Use only the final 26-feature table, frozen splits, compatible engineering definitions, and Notebook 06 retained combinations.

In [3]:
working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_run_directory = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
aggregation_directory = final_run_directory / "02_patient_level_aggregation"
split_directory = final_run_directory / "03_setup_and_splits"
feature_screen_directory = final_run_directory / "05_feature_pipeline_screening"
family_screen_directory = final_run_directory / "06_model_family_screening"
output_directory = final_run_directory / "07_tuning_and_outer_evaluation"
output_directory.mkdir(parents=True, exist_ok=True)

input_paths = {
    "predictors": aggregation_directory / "primary_1040_26_predictors.csv",
    "outcomes": aggregation_directory / "primary_1040_outcome_metadata.csv",
    "feature_manifest": split_directory / "primary_feature_manifest.csv",
    "outer_splits": split_directory / "outer_split_assignments.csv",
    "inner_folds": split_directory / "inner_fold_assignments.csv",
    "engineering_manifest": feature_screen_directory / "engineering_configuration_manifest.csv",
    "retained_combinations": family_screen_directory / "retained_for_focused_tuning.csv",
    "family_validation": family_screen_directory / "model_family_screening_validation.csv",
}
missing_inputs = [name for name, path in input_paths.items() if not path.is_file()]
assert not missing_inputs, f"Missing required inputs: {missing_inputs}"

print("Project root:", project_root)
print("Output directory:", output_directory.relative_to(project_root))

Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk
Output directory: results/final_pipeline/06_final_fit_and_performance_summary/07_tuning_and_outer_evaluation


## 4. Load and validate the frozen data

Confirm the 1,040-patient input, 26 source features, 20 paired splits, and two retained combinations for every seed and target.

In [4]:
predictors = pd.read_csv(
    input_paths["predictors"],
    dtype={"PATNO": "string"},
    low_memory=False,
).set_index("PATNO")
outcome_metadata = pd.read_csv(
    input_paths["outcomes"],
    dtype={"PATNO": "string"},
    usecols=["PATNO", "falls_class"],
)
feature_manifest = pd.read_csv(input_paths["feature_manifest"])
outer_assignments = pd.read_csv(
    input_paths["outer_splits"],
    dtype={"PATNO": "string"},
)
inner_assignments = pd.read_csv(
    input_paths["inner_folds"],
    dtype={"PATNO": "string"},
)
engineering_manifest = pd.read_csv(input_paths["engineering_manifest"])
retained = pd.read_csv(input_paths["retained_combinations"])
family_validation = pd.read_csv(input_paths["family_validation"])

features = feature_manifest["feature"].tolist()
outcomes = outcome_metadata.set_index("PATNO")["falls_class"].astype(int)
targets = ["direct", "stage_1", "stage_2"]
split_seeds = sorted(outer_assignments["split_seed"].unique())

retained_counts = retained.groupby(["split_seed", "target"]).size()
assert predictors.shape == (1040, 26)
assert predictors.columns.tolist() == features
assert set(predictors.index) == set(outcomes.index)
assert split_seeds == list(range(20))
assert len(retained_counts) == 60 and retained_counts.eq(2).all()
assert family_validation["passed"].astype(str).str.lower().eq("true").all()

print("Patients:", len(predictors))
print("Source features:", len(features))
print("Retained combinations:", len(retained))
display(retained.groupby(["target", "model_family"]).size().rename("retained_count").reset_index())

Patients: 1040
Source features: 26
Retained combinations: 120


,target,model_family,retained_count
0,direct,catboost,15
1,direct,extra_trees,2
2,direct,linear_svc,1
3,direct,random_forest,18
4,direct,rbf_svc,4
5,stage_1,catboost,4
6,stage_1,extra_trees,7
7,stage_1,hist_gradient_boosting,1
8,stage_1,linear_svc,2
9,stage_1,logistic,10


# Part A — Focused hyperparameter tuning

The next sections recreate the complete training-only pipeline and tune only the locally retained combinations. Every imputation value, encoding, scale, engineered representation, and selected feature is refitted inside each inner-training fold.

## 5. Recreate the frozen tuning grids

Use the grids locked before the earlier checkpointed tuning study. Grid size varies by model family, but no value is added after viewing an outer-test result.

In [5]:
MODEL_FAMILIES = [
    "logistic", "linear_svc", "rbf_svc", "random_forest", "extra_trees",
    "hist_gradient_boosting", "xgboost", "lightgbm", "catboost",
]
MODEL_ORDER = {family: order for order, family in enumerate(MODEL_FAMILIES)}
MODEL_BACKEND = {
    "logistic": "scaled_dense",
    "linear_svc": "scaled_dense",
    "rbf_svc": "scaled_dense",
    "random_forest": "unscaled_dense",
    "extra_trees": "unscaled_dense",
    "hist_gradient_boosting": "native",
    "xgboost": "native",
    "lightgbm": "native",
    "catboost": "native",
}

parameter_lists = {
    "logistic": [
        {"C": C, "class_weight": weight}
        for C in [0.1, 1.0, 10.0]
        for weight in [None, "balanced"]
    ],
    "linear_svc": [
        {"C": C, "class_weight": weight}
        for C in [0.1, 1.0, 10.0]
        for weight in [None, "balanced"]
    ],
    "rbf_svc": [
        {"C": C, "class_weight": weight, "gamma": gamma}
        for C in [0.1, 1.0, 10.0]
        for gamma in ["scale", 0.01, 0.1]
        for weight in [None, "balanced"]
    ],
    "random_forest": [
        {
            "n_estimators": 300,
            "max_depth": depth,
            "min_samples_leaf": leaf,
            "class_weight": weight,
        }
        for weight in [None, "balanced"]
        for depth in [None, 6]
        for leaf in [1, 5]
    ],
    "extra_trees": [
        {
            "n_estimators": 300,
            "max_depth": depth,
            "min_samples_leaf": leaf,
            "class_weight": weight,
        }
        for weight in [None, "balanced"]
        for depth in [None, 6]
        for leaf in [1, 5]
    ],
    "hist_gradient_boosting": [
        {
            "max_leaf_nodes": leaves,
            "learning_rate": rate,
            "l2_regularization": penalty,
            "class_weight": weight,
        }
        for weight in [None, "balanced"]
        for penalty in [0.0, 1.0]
        for rate in [0.05, 0.1]
        for leaves in [15, 31]
    ],
    "xgboost": [
        {
            "n_estimators": 300,
            "max_depth": depth,
            "learning_rate": rate,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": penalty,
            "sample_weight": weight,
        }
        for rate in [0.03, 0.08]
        for depth in [2, 4]
        for penalty in [1.0, 5.0]
        for weight in ["none", "balanced"]
    ],
    "lightgbm": [
        {
            "n_estimators": 300,
            "num_leaves": leaves,
            "learning_rate": rate,
            "min_child_samples": 20,
            "subsample": 0.8,
            "subsample_freq": 1,
            "colsample_bytree": 0.8,
            "reg_lambda": penalty,
            "class_weight": weight,
        }
        for leaves in [15, 31]
        for rate in [0.03, 0.08]
        for penalty in [1.0, 5.0]
        for weight in [None, "balanced"]
    ],
    "catboost": [
        {
            "iterations": iterations,
            "depth": depth,
            "learning_rate": rate,
            "l2_leaf_reg": 5.0,
            "auto_class_weights": weight,
        }
        for iterations in [300, 600]
        for depth in [4, 6]
        for rate in [0.03, 0.08]
        for weight in [None, "Balanced"]
    ],
}

tuning_grid = pd.DataFrame([
    {
        "model_family": family,
        "grid_id": grid_id,
        "parameters": json.dumps(parameters, sort_keys=True),
    }
    for family in MODEL_FAMILIES
    for grid_id, parameters in enumerate(parameter_lists[family])
])
assert len(tuning_grid) == 110
assert set(tuning_grid["model_family"]) == set(MODEL_FAMILIES)
display(tuning_grid.groupby("model_family").size().rename("grid_configurations").reset_index())

,model_family,grid_configurations
0,catboost,16
1,extra_trees,8
2,hist_gradient_boosting,16
3,lightgbm,16
4,linear_svc,6
5,logistic,6
6,random_forest,8
7,rbf_svc,18
8,xgboost,16


## 6. Define the approved clinical preprocessing

Recreate the final missing-state indicators and structural-zero rule before any fold-specific imputation or encoding.

In [6]:
NOMINAL_FEATURES = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}
ORDINAL_FEATURES = {
    "FRZGT12M", "SCAU14", "SCAU16", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "NP1CNST",
}
MISSING_INDICATORS = [
    "FOG_FORM_MISSING", "NQ_FORM_MISSING", "PART_IV_FORM_MISSING",
]


def representation_group(source):
    if source in {"FRZGT12M", "FOG_FORM_MISSING"}:
        return "GROUP_FREEZING_FORM"
    if source in {"NQ_GAUSSIAN_REVISION", "NQ_FORM_MISSING"}:
        return "GROUP_NEUROQOL_FORM"
    if source in {"NP4TOT", "PART_IV_FORM_MISSING"}:
        return "GROUP_PART_IV_FORM"
    return source


def clinical_frame(frame):
    raw = frame[features].copy()
    indicators = pd.DataFrame(index=raw.index)
    indicators["FOG_FORM_MISSING"] = raw["FRZGT12M"].isna().astype(int)
    indicators["NQ_FORM_MISSING"] = raw["NQ_GAUSSIAN_REVISION"].isna().astype(int)
    indicators["PART_IV_FORM_MISSING"] = raw["NP4TOT"].isna().astype(int)

    structural_zero = raw["NP4TOT"].isna() & raw["DOPTHERST"].eq("No")
    raw.loc[structural_zero, "NP4TOT"] = 0.0
    return pd.concat([raw, indicators], axis=1)

## 7. Define the corrected statistical selector

The corrected FDR branch tests observed training values only and groups encoded columns with their clinical source.

In [7]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(p_values[valid])]
    ranked = p_values[order] * len(valid) / np.arange(1, len(valid) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def numeric_p_value(values, target, ordinal):
    observed = pd.DataFrame({"value": values, "target": target}).dropna()
    groups = [
        observed.loc[observed["target"].eq(label), "value"].astype(float).to_numpy()
        for label in sorted(observed["target"].unique())
    ]
    if len(groups) < 2 or min(len(group) for group in groups) < 2:
        return np.nan
    if np.unique(np.concatenate(groups)).size < 2:
        return 1.0

    if ordinal:
        test = (
            stats.mannwhitneyu(*groups, alternative="two-sided")
            if len(groups) == 2
            else stats.kruskal(*groups)
        )
        return float(test.pvalue)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        normality = [
            stats.normaltest(group).pvalue
            if len(group) >= 8 and np.unique(group).size >= 3
            else 0.0
            for group in groups
        ]
        variance_p = stats.levene(*groups, center="median").pvalue
    parametric = (
        all(np.isfinite(normality))
        and min(normality) >= 0.05
        and np.isfinite(variance_p)
        and variance_p >= 0.05
    )
    if len(groups) == 2:
        test = (
            stats.ttest_ind(*groups, equal_var=True)
            if parametric
            else stats.mannwhitneyu(*groups, alternative="two-sided")
        )
    else:
        test = stats.f_oneway(*groups) if parametric else stats.kruskal(*groups)
    return float(test.pvalue)


def categorical_p_value(values, target, random_seed):
    categories = values.astype("string").fillna("Missing")
    table = pd.crosstab(categories, target)
    table = table.loc[table.sum(axis=1).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 1.0
    asymptotic = stats.chi2_contingency(table, correction=False)
    sparse = (
        asymptotic.expected_freq.min() < 1
        or (asymptotic.expected_freq < 5).mean() > 0.20
    )
    if not sparse:
        return float(asymptotic.pvalue)
    method = stats.PermutationMethod(
        n_resamples=999,
        rng=np.random.default_rng(random_seed),
    )
    return float(
        stats.chi2_contingency(
            table,
            correction=False,
            method=method,
        ).pvalue
    )


def select_groups_fdr(raw, target, random_seed):
    p_values = []
    for position, column in enumerate(raw.columns):
        if column in NOMINAL_FEATURES or column in MISSING_INDICATORS:
            p_value = categorical_p_value(
                raw[column],
                target,
                random_seed + position,
            )
        else:
            p_value = numeric_p_value(
                raw[column],
                target,
                ordinal=column in ORDINAL_FEATURES,
            )
        p_values.append(p_value)

    q_values = bh_adjust(p_values)
    selected_sources = [
        column
        for column, q_value in zip(raw.columns, q_values)
        if np.isfinite(q_value) and q_value <= 0.05
    ]
    if not selected_sources:
        finite = np.flatnonzero(np.isfinite(q_values))
        fallback = finite[np.argmin(q_values[finite])] if len(finite) else 0
        selected_sources = [raw.columns[fallback]]
    return {representation_group(source) for source in selected_sources}

## 8. Define the fold-fitted candidate transformer

This class keeps preprocessing, interaction/PCA construction, and feature selection inside the fitted pipeline. The longer cell is kept intact because these steps share fitted state.

In [8]:
engineering_definitions = {
    row.branch: {
        "kind": row.branch_kind,
        "sources": row.source_features.split(" | "),
        "threshold": None if pd.isna(row.variance_threshold) else float(row.variance_threshold),
    }
    for row in engineering_manifest.itertuples(index=False)
}


class CandidateTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        candidate_kind,
        selector,
        selector_parameter,
        branch=None,
        backend="scaled_dense",
        random_state=42,
    ):
        self.candidate_kind = candidate_kind
        self.selector = selector
        self.selector_parameter = selector_parameter
        self.branch = branch
        self.backend = backend
        self.random_state = random_state

    def _fit_preprocessing(self, raw):
        self.nominal_columns_ = [column for column in raw if column in NOMINAL_FEATURES]
        self.numeric_columns_ = [column for column in raw if column not in self.nominal_columns_]

        freezing_mode = raw["FRZGT12M"].mode(dropna=True)
        if freezing_mode.empty:
            raise ValueError("FRZGT12M has no observed training value.")
        self.freezing_fill_ = float(freezing_mode.iloc[0])

        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        self.medians_ = raw[median_columns].median()
        if self.medians_.isna().any():
            missing = self.medians_[self.medians_.isna()].index.tolist()
            raise ValueError(f"No training value available for: {missing}")

        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        self.scaler_ = StandardScaler().fit(dense_numeric)

        nominal = (
            raw[self.nominal_columns_]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )
        self.encoder_ = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ).fit(nominal)
        self.encoded_nominal_columns_ = self.encoder_.get_feature_names_out(
            self.nominal_columns_
        ).tolist()

        source_by_column = {column: column for column in self.numeric_columns_}
        encoded_sources = [
            source
            for source, categories in zip(
                self.nominal_columns_,
                self.encoder_.categories_,
            )
            for _ in categories
        ]
        source_by_column.update(
            dict(zip(self.encoded_nominal_columns_, encoded_sources))
        )
        self.group_by_column_ = {
            column: representation_group(source)
            for column, source in source_by_column.items()
        }

    def _matrices(self, raw):
        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        scaled_numeric = pd.DataFrame(
            self.scaler_.transform(dense_numeric),
            index=raw.index,
            columns=self.numeric_columns_,
        )

        nominal = (
            raw[self.nominal_columns_]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )
        encoded = pd.DataFrame(
            self.encoder_.transform(nominal),
            index=raw.index,
            columns=self.encoded_nominal_columns_,
        )
        native_numeric = raw[self.numeric_columns_].apply(
            pd.to_numeric,
            errors="coerce",
        )
        return {
            "scaled_dense": pd.concat([scaled_numeric, encoded], axis=1),
            "unscaled_dense": pd.concat([dense_numeric, encoded], axis=1),
            "native": pd.concat([native_numeric, encoded], axis=1),
        }

    def _apply_engineering(self, matrix, fit):
        if self.candidate_kind != "engineered_representation":
            return matrix
        definition = engineering_definitions[self.branch]
        matrix = matrix.copy()
        sources = definition["sources"]

        if definition["kind"] == "interaction":
            matrix[self.branch] = (
                matrix[sources[0]].to_numpy()
                * matrix[sources[1]].to_numpy()
            )
            return matrix

        if fit:
            self.pca_ = PCA(
                n_components=definition["threshold"],
                svd_solver="full",
            ).fit(matrix[sources])
        components = self.pca_.transform(matrix[sources])
        matrix = matrix.drop(columns=sources)
        names = [
            f"{self.branch}_PC{index + 1}"
            for index in range(components.shape[1])
        ]
        matrix[names] = components
        return matrix

    def _select_columns(self, raw, dense, target):
        if self.selector == "none":
            self.selected_groups_ = set(self.group_by_column_.values())
            return dense.columns.tolist()

        if self.selector == "corrected_fdr":
            selected_groups = select_groups_fdr(raw, target, self.random_state)
        elif self.selector == "l1":
            C_value = float(self.selector_parameter.split("=")[1])
            base_estimator = LogisticRegression(
                solver="liblinear",
                l1_ratio=1.0,
                C=C_value,
                class_weight="balanced",
                max_iter=5000,
                random_state=self.random_state,
            )
            selector = OneVsRestClassifier(base_estimator, n_jobs=-1)
            selector.fit(dense, target)
            importance = np.max(
                np.vstack([
                    np.abs(estimator.coef_).reshape(-1)
                    for estimator in selector.estimators_
                ]),
                axis=0,
            )
            selected_groups = {
                self.group_by_column_[column]
                for column, value in zip(dense.columns, importance)
                if value > 1e-10
            }
            if not selected_groups:
                strongest = dense.columns[int(np.argmax(importance))]
                selected_groups = {self.group_by_column_[strongest]}
        else:
            multiplier = 1.25 if "1.25" in self.selector_parameter else 1.0
            selector = ExtraTreesClassifier(
                n_estimators=120,
                max_depth=6,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=self.random_state,
                n_jobs=-1,
            )
            selector.fit(dense, target)
            group_importance = {}
            for column, value in zip(dense.columns, selector.feature_importances_):
                group = self.group_by_column_[column]
                group_importance[group] = group_importance.get(group, 0.0) + float(value)
            threshold = np.median(list(group_importance.values())) * multiplier
            selected_groups = {
                group
                for group, value in group_importance.items()
                if value >= threshold
            }
            if not selected_groups:
                selected_groups = {max(group_importance, key=group_importance.get)}

        self.selected_groups_ = selected_groups
        return [
            column
            for column in dense
            if self.group_by_column_[column] in selected_groups
        ]

    def fit(self, X, y):
        raw = clinical_frame(X)
        self._fit_preprocessing(raw)
        matrices = self._matrices(raw)
        dense = self._apply_engineering(matrices["scaled_dense"], fit=True)

        if self.candidate_kind == "engineered_representation":
            self.force_dense_ = True
            self.selected_columns_ = dense.columns.tolist()
            self.selected_groups_ = set(self.group_by_column_.values()) | {self.branch}
        else:
            self.force_dense_ = False
            self.selected_columns_ = self._select_columns(raw, dense, y)
        if not self.selected_columns_:
            raise RuntimeError("Candidate transformer selected no columns.")
        return self

    def transform(self, X):
        raw = clinical_frame(X)
        matrices = self._matrices(raw)
        if self.force_dense_:
            output = self._apply_engineering(
                matrices["scaled_dense"],
                fit=False,
            )
        else:
            output = matrices[self.backend]
        return output[self.selected_columns_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_columns_, dtype=object)

## 9. Build models and target-specific outcomes

Construct each tuned estimator, restrict Stage 2 training to true fallers, and calculate the selection metrics.

In [9]:
def build_model(family, target_name, random_seed, parameters, probability=False):
    parameters = dict(parameters)
    if family == "logistic":
        return LogisticRegression(
            max_iter=5000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "linear_svc":
        return LinearSVC(
            dual="auto",
            max_iter=10000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "rbf_svc":
        return SVC(
            cache_size=1000,
            probability=probability,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "random_forest":
        return RandomForestClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "extra_trees":
        return ExtraTreesClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            random_state=random_seed,
            **parameters,
        ), None
    if family == "xgboost":
        weight_mode = parameters.pop("sample_weight")
        class_count = 3 if target_name == "direct" else 2
        parameters.update({
            "objective": "multi:softprob" if class_count == 3 else "binary:logistic",
            "eval_metric": "mlogloss" if class_count == 3 else "logloss",
            "random_state": random_seed,
            "n_jobs": -1,
            "verbosity": 0,
        })
        if class_count == 3:
            parameters["num_class"] = 3
        return XGBClassifier(**parameters), weight_mode
    if family == "lightgbm":
        return LGBMClassifier(
            random_state=random_seed,
            n_jobs=-1,
            verbosity=-1,
            **parameters,
        ), None

    loss = "MultiClass" if target_name == "direct" else "Logloss"
    return CatBoostClassifier(
        random_seed=random_seed,
        loss_function=loss,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **parameters,
    ), None


def target_data(target_name, training_ids, validation_ids):
    y_train = outcomes.loc[list(training_ids)]
    y_validation = outcomes.loc[list(validation_ids)]
    if target_name == "direct":
        return list(training_ids), list(validation_ids), y_train, y_validation
    if target_name == "stage_1":
        return (
            list(training_ids),
            list(validation_ids),
            y_train.gt(0).astype(int),
            y_validation.gt(0).astype(int),
        )
    y_train = y_train[y_train.gt(0)]
    y_validation = y_validation[y_validation.gt(0)]
    return (
        y_train.index.tolist(),
        y_validation.index.tolist(),
        y_train.eq(2).astype(int),
        y_validation.eq(2).astype(int),
    )


def classification_metrics(target_name, truth, predictions):
    labels = [0, 1, 2] if target_name == "direct" else [0, 1]
    recalls = recall_score(
        truth,
        predictions,
        labels=labels,
        average=None,
        zero_division=0,
    )
    result = {
        "macro_f1": f1_score(truth, predictions, labels=labels, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "accuracy": accuracy_score(truth, predictions),
        "recall_class_0": recalls[0],
        "recall_class_1": recalls[1],
        "recall_class_2": recalls[2] if target_name == "direct" else np.nan,
    }
    result["priority_recall"] = (
        result["recall_class_0"]
        if target_name == "stage_2"
        else result["recall_class_1"]
    )
    return result

## 10. Treat model warnings as actionable failures

Capture warnings during fitting so the notebook stops with one concise message instead of producing pages of warning output.

In [10]:
def fit_checked(estimator, X, y, context, **fit_parameters):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        estimator.fit(X, y, **fit_parameters)

    if caught:
        unique_messages = list(dict.fromkeys(str(item.message) for item in caught))
        convergence = any(
            issubclass(item.category, ConvergenceWarning)
            for item in caught
        )
        warning_type = "convergence warning" if convergence else "model warning"
        raise RuntimeError(
            f"{context} emitted a {warning_type}: {unique_messages[0]}"
        )
    return estimator

## 11. Lock the run identity

Hash every upstream input and the generated frozen grid. Existing checkpoints can resume only when the configuration is unchanged.

In [11]:
RUN_VERSION = "final-notebook-07-focused-tuning-and-outer-evaluation-v2-26-features-d28"


def file_digest(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def frame_digest(frame):
    canonical = frame.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(canonical.encode()).hexdigest()


identity = {
    "run_version": RUN_VERSION,
    "input_sha256": {
        name: file_digest(path)
        for name, path in input_paths.items()
    },
    "tuning_grid_sha256": frame_digest(tuning_grid),
    "targets": targets,
    "inner_folds": 5,
    "selection_metric": "mean inner-fold macro F1",
    "near_tie_margin": 0.01,
    "outer_test_use": "one evaluation after local winner selection",
}
configuration_hash = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode()
).hexdigest()
manifest_path = output_directory / "run_manifest.json"
manifest = {
    **identity,
    "configuration_hash": configuration_hash,
    "python": platform.python_version(),
    "scikit_learn": sklearn_version,
    "xgboost": xgboost_version,
    "lightgbm": lightgbm_version,
    "catboost": catboost_version,
}
if manifest_path.is_file():
    existing = json.loads(manifest_path.read_text())
    if existing.get("configuration_hash") != configuration_hash:
        raise RuntimeError(
            "Existing checkpoints belong to a different configuration. "
            "Preserve them and use a new output directory."
        )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

print("Configuration hash:", configuration_hash[:16])
print(pd.Series({
    name: manifest[name]
    for name in ["python", "scikit_learn", "xgboost", "lightgbm", "catboost"]
}).to_string())

Configuration hash: a81970599fbd5adc
python          3.13.12
scikit_learn      1.8.0
xgboost           3.2.0
lightgbm          4.7.0
catboost         1.2.10


## 12. Show the focused tuning workload

Count the exact number of inner-fold fits. Each checkpoint unit is one retained combination, so an interrupted run resumes after the last complete unit.

In [12]:
grid_sizes = tuning_grid.groupby("model_family").size()
work_plan = retained[[
    "split_seed", "target", "retained_rank", "model_family", "mean_fit_seconds",
]].copy()
work_plan["grid_configurations"] = work_plan["model_family"].map(grid_sizes)
work_plan["planned_fits"] = work_plan["grid_configurations"] * 5
work_plan["rough_minutes"] = (
    work_plan["planned_fits"] * work_plan["mean_fit_seconds"] / 60
)

family_work = work_plan.groupby("model_family", as_index=False).agg(
    retained_units=("retained_rank", "size"),
    grid_configurations=("grid_configurations", "first"),
    planned_fits=("planned_fits", "sum"),
    rough_minutes=("rough_minutes", "sum"),
)
display(family_work)
print("Resumable units:", len(work_plan))
print(f"Planned inner-fold fits: {int(work_plan['planned_fits'].sum()):,}")
print(
    "Rough model-fit estimate:",
    f"{work_plan['rough_minutes'].sum():.1f} minutes plus preprocessing and checkpoint overhead",
)

,model_family,retained_units,grid_configurations,planned_fits,rough_minutes
0,catboost,22,16,1760,6.966798
1,extra_trees,15,8,600,2.167888
2,hist_gradient_boosting,4,16,320,1.856900
3,lightgbm,3,16,240,0.777894
4,linear_svc,6,6,180,0.432297
5,logistic,14,6,420,0.743450
6,random_forest,40,8,1600,8.055001
7,rbf_svc,15,18,1350,2.623003
8,xgboost,1,16,80,0.260542


Resumable units: 120
Planned inner-fold fits: 6,550
Rough model-fit estimate: 23.9 minutes plus preprocessing and checkpoint overhead


## 13. Run or resume focused tuning

For each retained combination, evaluate every frozen hyperparameter row on the same five inner folds. Progress and estimated time remaining are shown after every checkpoint unit.

In [13]:
def atomic_csv(frame, file_path):
    temporary = file_path.with_suffix(file_path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(file_path)


checkpoint_path = output_directory / "focused_tuning_checkpoint.csv"
unit_columns = ["split_seed", "target", "retained_rank"]
saved = pd.read_csv(checkpoint_path) if checkpoint_path.is_file() else pd.DataFrame()
complete_units = set()
if not saved.empty:
    for key, group in saved.groupby(unit_columns):
        family = group["model_family"].iloc[0]
        expected = int(grid_sizes.loc[family]) * 5
        if len(group) == expected:
            complete_units.add(tuple(key))

if saved.empty:
    tuning_rows = []
else:
    keep = saved.apply(
        lambda row: (
            int(row["split_seed"]), row["target"], int(row["retained_rank"])
        ) in complete_units,
        axis=1,
    )
    tuning_rows = saved.loc[keep].to_dict("records")

planned_total_fits = int(work_plan["planned_fits"].sum())
completed_at_start = len(tuning_rows)
session_start = time.perf_counter()
print(
    f"Resuming with {len(complete_units)}/{len(work_plan)} units and "
    f"{completed_at_start:,}/{planned_total_fits:,} fits complete.",
    flush=True,
)

ordered_units = retained.sort_values(["split_seed", "target", "retained_rank"])
for retained_row in ordered_units.itertuples(index=False):
    unit = (
        int(retained_row.split_seed),
        retained_row.target,
        int(retained_row.retained_rank),
    )
    if unit in complete_units:
        continue

    family_grid = tuning_grid.loc[
        tuning_grid["model_family"].eq(retained_row.model_family)
    ].sort_values("grid_id")
    seed_inner = inner_assignments.loc[
        inner_assignments["split_seed"].eq(retained_row.split_seed)
    ]
    outer_train_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(retained_row.split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ]
    )
    outer_test_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(retained_row.split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ]
    )
    assert set(seed_inner["PATNO"]) == outer_train_ids
    assert outer_test_ids.isdisjoint(outer_train_ids)

    branch = None if pd.isna(retained_row.branch) else retained_row.branch
    unit_rows = []
    unit_start = time.perf_counter()
    print(
        f"Starting seed {retained_row.split_seed:02d}, {retained_row.target}, "
        f"retained rank {retained_row.retained_rank}: {retained_row.model_family}, "
        f"{len(family_grid)} configurations × 5 folds",
        flush=True,
    )

    for grid_row in family_grid.itertuples(index=False):
        parameters = json.loads(grid_row.parameters)
        for inner_fold in range(5):
            validation_ids = sorted(
                seed_inner.loc[
                    seed_inner["inner_validation_fold"].eq(inner_fold),
                    "PATNO",
                ].tolist()
            )
            training_ids = sorted(outer_train_ids - set(validation_ids))
            assert set(training_ids).isdisjoint(validation_ids)
            assert outer_test_ids.isdisjoint([*training_ids, *validation_ids])

            train_ids, valid_ids, y_train, y_validation = target_data(
                retained_row.target,
                training_ids,
                validation_ids,
            )
            random_seed = 400_000 + int(retained_row.split_seed) * 10 + inner_fold
            transformer_instance = CandidateTransformer(
                candidate_kind=retained_row.candidate_kind,
                selector=retained_row.selector,
                selector_parameter=retained_row.selector_parameter,
                branch=branch,
                backend=MODEL_BACKEND[retained_row.model_family],
                random_state=random_seed,
            )
            estimator, weight_mode = build_model(
                retained_row.model_family,
                retained_row.target,
                random_seed,
                parameters,
            )
            pipeline = Pipeline([
                ("candidate", transformer_instance),
                ("model", estimator),
            ])
            fit_parameters = {}
            if weight_mode == "balanced":
                fit_parameters["model__sample_weight"] = compute_sample_weight(
                    "balanced",
                    y_train,
                )

            fit_start = time.perf_counter()
            fit_checked(
                pipeline,
                predictors.loc[train_ids],
                y_train,
                context=(
                    f"seed {retained_row.split_seed}, {retained_row.target}, "
                    f"rank {retained_row.retained_rank}, grid {grid_row.grid_id}, "
                    f"fold {inner_fold}"
                ),
                **fit_parameters,
            )
            predictions = pipeline.predict(predictors.loc[valid_ids])
            fitted_transformer = pipeline.named_steps["candidate"]
            unit_rows.append({
                "split_seed": int(retained_row.split_seed),
                "target": retained_row.target,
                "retained_rank": int(retained_row.retained_rank),
                "configuration_id": retained_row.configuration_id,
                "pipeline_key": retained_row.pipeline_key,
                "candidate_kind": retained_row.candidate_kind,
                "selector": retained_row.selector,
                "selector_parameter": retained_row.selector_parameter,
                "branch": branch,
                "branch_kind": retained_row.branch_kind,
                "model_family": retained_row.model_family,
                "backend": (
                    "scaled_dense"
                    if fitted_transformer.force_dense_
                    else MODEL_BACKEND[retained_row.model_family]
                ),
                "grid_id": int(grid_row.grid_id),
                "parameters": grid_row.parameters,
                "inner_validation_fold": inner_fold,
                "training_patients": len(y_train),
                "validation_patients": len(y_validation),
                "selected_columns": len(fitted_transformer.selected_columns_),
                "selected_groups": len(fitted_transformer.selected_groups_),
                "selected_column_names": " | ".join(fitted_transformer.selected_columns_),
                "fit_seconds": time.perf_counter() - fit_start,
                **classification_metrics(
                    retained_row.target,
                    y_validation,
                    predictions,
                ),
            })

    expected = len(family_grid) * 5
    assert len(unit_rows) == expected
    tuning_rows.extend(unit_rows)
    complete_units.add(unit)
    atomic_csv(pd.DataFrame(tuning_rows), checkpoint_path)

    completed_fits = len(tuning_rows)
    session_elapsed = time.perf_counter() - session_start
    session_completed = completed_fits - completed_at_start
    remaining_fits = planned_total_fits - completed_fits
    eta_minutes = (
        remaining_fits * session_elapsed / session_completed / 60
        if session_completed
        else np.nan
    )
    print(
        f"Completed {len(complete_units)}/{len(work_plan)} units and "
        f"{completed_fits:,}/{planned_total_fits:,} fits; "
        f"unit time {(time.perf_counter() - unit_start) / 60:.1f} min; "
        f"estimated remaining {eta_minutes:.1f} min",
        flush=True,
    )

tuning_folds = pd.DataFrame(tuning_rows).sort_values(
    ["split_seed", "target", "retained_rank", "grid_id", "inner_validation_fold"]
).reset_index(drop=True)
print(f"Focused tuning complete: {len(tuning_folds):,} inner-fold results.")

Resuming with 0/120 units and 0/6,550 fits complete.
Starting seed 00, direct, retained rank 1: catboost, 16 configurations × 5 folds
Completed 1/120 units and 80/6,550 fits; unit time 0.9 min; estimated remaining 70.5 min
Starting seed 00, direct, retained rank 2: catboost, 16 configurations × 5 folds
Completed 2/120 units and 160/6,550 fits; unit time 0.8 min; estimated remaining 66.3 min
Starting seed 00, stage_1, retained rank 1: rbf_svc, 18 configurations × 5 folds
Completed 3/120 units and 250/6,550 fits; unit time 0.1 min; estimated remaining 43.2 min
Starting seed 00, stage_1, retained rank 2: rbf_svc, 18 configurations × 5 folds
Completed 4/120 units and 340/6,550 fits; unit time 0.1 min; estimated remaining 32.3 min
Starting seed 00, stage_2, retained rank 1: lightgbm, 16 configurations × 5 folds
Completed 5/120 units and 420/6,550 fits; unit time 0.2 min; estimated remaining 28.8 min
Starting seed 00, stage_2, retained rank 2: random_forest, 8 configurations × 5 folds
Comple

## 14. Select one tuned winner per seed and target

Choose by mean inner-fold macro F1. Among configurations within 0.01 of the best, prioritize the target-specific recall, then fewer retained groups and columns, then the prespecified simpler family order.

In [14]:
summary_columns = [
    "split_seed", "target", "retained_rank", "configuration_id", "pipeline_key",
    "candidate_kind", "selector", "selector_parameter", "branch", "branch_kind",
    "model_family", "backend", "grid_id", "parameters",
]
tuning_summary = tuning_folds.groupby(
    summary_columns,
    dropna=False,
    as_index=False,
).agg(
    mean_macro_f1=("macro_f1", "mean"),
    sd_macro_f1=("macro_f1", "std"),
    mean_priority_recall=("priority_recall", "mean"),
    mean_balanced_accuracy=("balanced_accuracy", "mean"),
    mean_accuracy=("accuracy", "mean"),
    mean_selected_columns=("selected_columns", "mean"),
    mean_selected_groups=("selected_groups", "mean"),
    mean_fit_seconds=("fit_seconds", "mean"),
    inner_folds=("inner_validation_fold", "nunique"),
)


def choose_winner(group):
    best_macro_f1 = group["mean_macro_f1"].max()
    near_ties = group.loc[
        group["mean_macro_f1"].ge(best_macro_f1 - 0.01)
    ].copy()
    near_ties["model_order"] = near_ties["model_family"].map(MODEL_ORDER)
    return near_ties.sort_values(
        [
            "mean_priority_recall", "mean_selected_groups", "mean_selected_columns",
            "model_order", "grid_id", "configuration_id",
        ],
        ascending=[False, True, True, True, True, True],
    ).iloc[0].drop(labels="model_order")


winners = pd.DataFrame([
    choose_winner(group)
    for _, group in tuning_summary.groupby(
        ["split_seed", "target"],
        sort=False,
    )
]).reset_index(drop=True)
winner_frequency = (
    winners.groupby(["target", "model_family"])
    .size()
    .rename("winner_count")
    .reset_index()
    .sort_values(["target", "winner_count"], ascending=[True, False])
)
display(winner_frequency)
print("Partition-specific tuned winners:", len(winners))

,target,model_family,winner_count
3,direct,random_forest,9
0,direct,catboost,7
4,direct,rbf_svc,2
1,direct,extra_trees,1
2,direct,linear_svc,1
6,stage_1,extra_trees,6
9,stage_1,rbf_svc,5
7,stage_1,logistic,4
8,stage_1,random_forest,3
5,stage_1,catboost,2


Partition-specific tuned winners: 60


## 15. Validate the tuning stage

Verify complete grid coverage, five-fold pairing, one local winner per seed and target, and continued outer-test isolation.

In [15]:
expected_by_unit = {
    (int(row.split_seed), row.target, int(row.retained_rank)): int(
        grid_sizes.loc[row.model_family]
    ) * 5
    for row in retained.itertuples(index=False)
}
actual_by_unit = tuning_folds.groupby(unit_columns).size().to_dict()
active_grid_rows = {
    (family, int(grid_id))
    for family in retained["model_family"].unique()
    for grid_id in tuning_grid.loc[
        tuning_grid["model_family"].eq(family),
        "grid_id",
    ]
}
used_grid_rows = set(zip(
    tuning_folds["model_family"],
    tuning_folds["grid_id"].astype(int),
))
metric_columns = ["macro_f1", "priority_recall", "balanced_accuracy", "accuracy"]

tuning_validation = pd.DataFrame([
    {
        "check": "all 120 retained-combination units complete",
        "passed": actual_by_unit == expected_by_unit,
        "detail": "one checkpoint unit per retained combination",
    },
    {
        "check": "planned inner-fold fits complete",
        "passed": len(tuning_folds) == planned_total_fits,
        "detail": f"{planned_total_fits:,} fits",
    },
    {
        "check": "every active frozen grid row evaluated",
        "passed": used_grid_rows == active_grid_rows,
        "detail": "only families retained by Notebook 06 are active",
    },
    {
        "check": "five folds per candidate and grid row",
        "passed": tuning_folds.groupby(
            unit_columns + ["grid_id"]
        )["inner_validation_fold"].nunique().eq(5).all(),
        "detail": "paired inner validation",
    },
    {
        "check": "all fitted pipelines retained columns",
        "passed": tuning_folds["selected_columns"].gt(0).all(),
        "detail": "no empty model input",
    },
    {
        "check": "all tuning metrics are bounded",
        "passed": tuning_folds[metric_columns].apply(
            lambda column: column.between(0, 1).all()
        ).all(),
        "detail": "inner-fold metrics",
    },
    {
        "check": "one tuned winner per seed and target",
        "passed": len(winners) == 60 and winners.groupby(
            ["split_seed", "target"]
        ).size().eq(1).all(),
        "detail": "20 seeds × 3 targets",
    },
    {
        "check": "no outer-test result used during tuning",
        "passed": not any(
            "outer_prediction" in column or "outer_metric" in column
            for column in tuning_folds.columns
        ),
        "detail": "inner validation only",
    },
    {
        "check": "checkpoint identity matches",
        "passed": json.loads(manifest_path.read_text())["configuration_hash"]
        == configuration_hash,
        "detail": "input, grid, and software identity",
    },
])
display(tuning_validation)
assert tuning_validation["passed"].all(), tuning_validation.loc[
    ~tuning_validation["passed"]
]
print(
    f"Tuning validation passed: {tuning_validation['passed'].sum()}/"
    f"{len(tuning_validation)}"
)

,check,passed,detail
0,all 120 retained-combination units complete,True,one checkpoint unit per retained combination
1,planned inner-fold fits complete,True,"6,550 fits"
2,every active frozen grid row evaluated,True,only families retained by Notebook 06 are active
3,five folds per candidate and grid row,True,paired inner validation
4,all fitted pipelines retained columns,True,no empty model input
5,all tuning metrics are bounded,True,inner-fold metrics
6,one tuned winner per seed and target,True,20 seeds × 3 targets
7,no outer-test result used during tuning,True,inner validation only
8,checkpoint identity matches,True,"input, grid, and software identity"


Tuning validation passed: 9/9


## 16. Save the locked tuning results

Write the grid, fold-level tuning results, summaries, and 60 local winners before any outer-test prediction is made.

In [16]:
def frames_equivalent(current, existing):
    if current.columns.tolist() != existing.columns.tolist() or current.shape != existing.shape:
        return False
    for column in current.columns:
        left = current[column]
        right = existing[column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        left_numeric_ok = left_numeric.notna().eq(left.notna()).all()
        right_numeric_ok = right_numeric.notna().eq(right.notna()).all()
        if left_numeric_ok and right_numeric_ok:
            if not np.allclose(
                left_numeric.to_numpy(dtype=float),
                right_numeric.to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            ):
                return False
        else:
            left_text = left.astype("string").fillna("<NA>").reset_index(drop=True)
            right_text = right.astype("string").fillna("<NA>").reset_index(drop=True)
            if not left_text.equals(right_text):
                return False
    return True


def save_new_or_equivalent(frame, file_path):
    if file_path.is_file():
        existing = pd.read_csv(file_path, low_memory=False)
        if not frames_equivalent(frame.reset_index(drop=True), existing):
            raise FileExistsError(
                f"Existing artifact truly differs and was not overwritten: {file_path.name}"
            )
        return "already equivalent"
    frame.to_csv(file_path, index=False)
    return "created"


tuning_artifacts = {
    "tuning_grid_manifest.csv": tuning_grid,
    "focused_tuning_inner_fold_results.csv": tuning_folds,
    "focused_tuning_partition_summary.csv": tuning_summary,
    "tuned_partition_winners.csv": winners,
    "tuned_winner_family_frequency.csv": winner_frequency,
    "focused_tuning_validation.csv": tuning_validation,
}
tuning_save_rows = []
for filename, frame in tuning_artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    tuning_save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})
display(pd.DataFrame(tuning_save_rows))
print("The local winners are now frozen for one-time outer evaluation.")

,artifact,status,rows
0,tuning_grid_manifest.csv,created,110
1,focused_tuning_inner_fold_results.csv,created,6550
2,focused_tuning_partition_summary.csv,created,1310
3,tuned_partition_winners.csv,created,60
4,tuned_winner_family_frequency.csv,created,19
5,focused_tuning_validation.csv,created,9


The local winners are now frozen for one-time outer evaluation.


# Part B — One-time outer evaluation

The following sections use the 60 frozen winners. For each seed, the direct and two-stage systems share the same 728 training patients and 312 test patients.

## 17. Define probability and locked-fit helpers

Use model probabilities when available. If Linear SVC wins, convert its decision scores with a fixed sigmoid/softmax only for AUC and two-stage score composition; this conversion uses no test-fitted quantity.

In [17]:
def aligned_score_matrix(pipeline, frame, labels):
    model = pipeline.named_steps["model"]
    if hasattr(model, "predict_proba"):
        raw = np.asarray(pipeline.predict_proba(frame), dtype=float)
        score_source = "predict_proba"
    else:
        decision = np.asarray(pipeline.decision_function(frame), dtype=float)
        if decision.ndim == 1:
            positive = 1.0 / (1.0 + np.exp(-np.clip(decision, -700, 700)))
            raw = np.column_stack([1.0 - positive, positive])
        else:
            shifted = decision - decision.max(axis=1, keepdims=True)
            exponentiated = np.exp(shifted)
            raw = exponentiated / exponentiated.sum(axis=1, keepdims=True)
        score_source = "normalized_decision_score"

    aligned = np.zeros((len(raw), len(labels)), dtype=float)
    classes = np.asarray(model.classes_).astype(int)
    for source_column, label in enumerate(classes):
        aligned[:, labels.index(int(label))] = raw[:, source_column]
    assert np.isfinite(aligned).all()
    assert np.allclose(aligned.sum(axis=1), 1.0)
    return aligned, score_source


def fit_locked_pipeline(winner, training_ids, test_ids, random_seed):
    target_name = winner.target
    y_original = outcomes.loc[training_ids]
    model_training_ids = list(training_ids)
    if target_name == "direct":
        y_train = y_original
        labels = [0, 1, 2]
    elif target_name == "stage_1":
        y_train = y_original.gt(0).astype(int)
        labels = [0, 1]
    else:
        y_train = y_original[y_original.gt(0)].eq(2).astype(int)
        model_training_ids = y_train.index.tolist()
        labels = [0, 1]

    branch = None if pd.isna(winner.branch) else winner.branch
    transformer_instance = CandidateTransformer(
        candidate_kind=winner.candidate_kind,
        selector=winner.selector,
        selector_parameter=winner.selector_parameter,
        branch=branch,
        backend=MODEL_BACKEND[winner.model_family],
        random_state=random_seed,
    )
    estimator, weight_mode = build_model(
        winner.model_family,
        target_name,
        random_seed,
        json.loads(winner.parameters),
        probability=winner.model_family == "rbf_svc",
    )
    pipeline = Pipeline([
        ("candidate", transformer_instance),
        ("model", estimator),
    ])
    fit_parameters = {}
    if weight_mode == "balanced":
        fit_parameters["model__sample_weight"] = compute_sample_weight(
            "balanced",
            y_train,
        )

    fit_start = time.perf_counter()
    fit_checked(
        pipeline,
        predictors.loc[model_training_ids],
        y_train,
        context=f"outer refit seed {winner.split_seed}, {target_name}",
        **fit_parameters,
    )
    test_frame = predictors.loc[test_ids]
    predictions = np.asarray(pipeline.predict(test_frame)).astype(int).reshape(-1)
    scores, score_source = aligned_score_matrix(pipeline, test_frame, labels)
    fitted_transformer = pipeline.named_steps["candidate"]
    return {
        "predictions": predictions,
        "scores": scores,
        "score_source": score_source,
        "selected_columns": fitted_transformer.selected_columns_,
        "selected_groups": fitted_transformer.selected_groups_,
        "training_patients": len(model_training_ids),
        "fit_seconds": time.perf_counter() - fit_start,
    }

## 18. Confirm the outer workload

Each seed requires three locked fits. The two reported systems then predict the same 312 held-out patients.

In [18]:
outer_work = pd.DataFrame({
    "outer_seeds": [len(split_seeds)],
    "locked_pipeline_fits": [len(split_seeds) * 3],
    "test_patients_per_seed": [312],
    "reported_systems": [2],
    "checkpoint_boundary": ["one split seed"],
})
display(outer_work)
print("Outer scores will not be used to revise any winner.")

,outer_seeds,locked_pipeline_fits,test_patients_per_seed,reported_systems,checkpoint_boundary
0,20,60,312,2,one split seed


Outer scores will not be used to revise any winner.


## 19. Run or resume the outer evaluation

Refit the three locked pipelines on all 728 training patients for a seed, create direct and hard-routed two-stage predictions, and checkpoint that seed.

In [19]:
prediction_checkpoint = output_directory / "outer_prediction_checkpoint.csv"
component_checkpoint = output_directory / "component_prediction_checkpoint.csv"
selected_checkpoint = output_directory / "outer_selected_pipeline_checkpoint.csv"
completed_path = output_directory / "completed_outer_seeds.csv"

if completed_path.is_file():
    completed_outer_seeds = set(
        pd.read_csv(completed_path)["split_seed"].astype(int).tolist()
    )
else:
    completed_outer_seeds = set()

prediction_rows = (
    pd.read_csv(prediction_checkpoint, dtype={"PATNO": "string"}).to_dict("records")
    if prediction_checkpoint.is_file()
    else []
)
component_rows = (
    pd.read_csv(component_checkpoint, dtype={"PATNO": "string"}).to_dict("records")
    if component_checkpoint.is_file()
    else []
)
selected_rows = (
    pd.read_csv(selected_checkpoint).to_dict("records")
    if selected_checkpoint.is_file()
    else []
)
prediction_rows = [
    row for row in prediction_rows
    if int(row["split_seed"]) in completed_outer_seeds
]
component_rows = [
    row for row in component_rows
    if int(row["split_seed"]) in completed_outer_seeds
]
selected_rows = [
    row for row in selected_rows
    if int(row["split_seed"]) in completed_outer_seeds
]

outer_session_times = []
print(
    f"Resuming with {len(completed_outer_seeds)}/{len(split_seeds)} outer seeds complete.",
    flush=True,
)
for split_seed in split_seeds:
    if split_seed in completed_outer_seeds:
        continue

    seed_start = time.perf_counter()
    training_ids = sorted(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ].tolist()
    )
    test_ids = sorted(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ].tolist()
    )
    assert len(training_ids) == 728 and len(test_ids) == 312
    assert set(training_ids).isdisjoint(test_ids)
    assert set(training_ids) | set(test_ids) == set(predictors.index)

    chosen = {
        target: winners.loc[
            winners["split_seed"].eq(split_seed)
            & winners["target"].eq(target)
        ].iloc[0]
        for target in targets
    }
    random_seed = 500_000 + int(split_seed)
    fitted = {
        target: fit_locked_pipeline(
            chosen[target],
            training_ids,
            test_ids,
            random_seed,
        )
        for target in targets
    }

    direct_prediction = fitted["direct"]["predictions"]
    direct_scores = fitted["direct"]["scores"]
    stage_1_prediction = fitted["stage_1"]["predictions"]
    stage_1_scores = fitted["stage_1"]["scores"]
    stage_2_prediction = fitted["stage_2"]["predictions"]
    stage_2_scores = fitted["stage_2"]["scores"]
    two_stage_prediction = np.where(
        stage_1_prediction == 0,
        0,
        stage_2_prediction + 1,
    )
    two_stage_scores = np.column_stack([
        stage_1_scores[:, 0],
        stage_1_scores[:, 1] * stage_2_scores[:, 0],
        stage_1_scores[:, 1] * stage_2_scores[:, 1],
    ])
    assert np.allclose(two_stage_scores.sum(axis=1), 1.0)
    y_test = outcomes.loc[test_ids].to_numpy()

    new_predictions = []
    for system, predicted, scores in [
        ("direct", direct_prediction, direct_scores),
        ("two_stage", two_stage_prediction, two_stage_scores),
    ]:
        new_predictions.extend({
            "split_seed": int(split_seed),
            "PATNO": patient,
            "system": system,
            "true_class": int(truth),
            "predicted_class": int(label),
            "score_no_fall": float(score[0]),
            "score_rare_fall": float(score[1]),
            "score_recurrent_fall": float(score[2]),
        } for patient, truth, label, score in zip(
            test_ids,
            y_test,
            predicted,
            scores,
        ))

    new_components = []
    for patient, truth, stage_1_label, stage_1_score, stage_2_label, stage_2_score in zip(
        test_ids,
        y_test,
        stage_1_prediction,
        stage_1_scores,
        stage_2_prediction,
        stage_2_scores,
    ):
        new_components.append({
            "split_seed": int(split_seed),
            "PATNO": patient,
            "target": "stage_1",
            "true_class": int(truth > 0),
            "predicted_class": int(stage_1_label),
            "score_class_0": float(stage_1_score[0]),
            "score_class_1": float(stage_1_score[1]),
            "eligible_for_component_metric": True,
        })
        new_components.append({
            "split_seed": int(split_seed),
            "PATNO": patient,
            "target": "stage_2",
            "true_class": int(truth == 2),
            "predicted_class": int(stage_2_label),
            "score_class_0": float(stage_2_score[0]),
            "score_class_1": float(stage_2_score[1]),
            "eligible_for_component_metric": bool(truth > 0),
        })

    new_selected = []
    for target_name in targets:
        winner = chosen[target_name]
        new_selected.append({
            "split_seed": int(split_seed),
            "target": target_name,
            "configuration_id": winner.configuration_id,
            "pipeline_key": winner.pipeline_key,
            "candidate_kind": winner.candidate_kind,
            "selector": winner.selector,
            "selector_parameter": winner.selector_parameter,
            "branch": None if pd.isna(winner.branch) else winner.branch,
            "branch_kind": winner.branch_kind,
            "model_family": winner.model_family,
            "grid_id": int(winner.grid_id),
            "parameters": winner.parameters,
            "score_source": fitted[target_name]["score_source"],
            "training_patients": fitted[target_name]["training_patients"],
            "selected_group_count": len(fitted[target_name]["selected_groups"]),
            "selected_groups": " | ".join(sorted(fitted[target_name]["selected_groups"])),
            "selected_column_count": len(fitted[target_name]["selected_columns"]),
            "selected_column_names": " | ".join(fitted[target_name]["selected_columns"]),
            "fit_seconds": fitted[target_name]["fit_seconds"],
        })

    prediction_rows.extend(new_predictions)
    component_rows.extend(new_components)
    selected_rows.extend(new_selected)
    atomic_csv(pd.DataFrame(prediction_rows), prediction_checkpoint)
    atomic_csv(pd.DataFrame(component_rows), component_checkpoint)
    atomic_csv(pd.DataFrame(selected_rows), selected_checkpoint)
    completed_outer_seeds.add(int(split_seed))
    atomic_csv(
        pd.DataFrame({"split_seed": sorted(completed_outer_seeds)}),
        completed_path,
    )

    elapsed = time.perf_counter() - seed_start
    outer_session_times.append(elapsed)
    remaining = len(split_seeds) - len(completed_outer_seeds)
    eta_minutes = remaining * np.mean(outer_session_times) / 60
    print(
        f"Completed {len(completed_outer_seeds)}/{len(split_seeds)} outer seeds; "
        f"seed {split_seed:02d} took {elapsed / 60:.1f} min; "
        f"estimated remaining {eta_minutes:.1f} min",
        flush=True,
    )

outer_predictions = pd.DataFrame(prediction_rows).sort_values(
    ["split_seed", "system", "PATNO"]
).reset_index(drop=True)
component_predictions = pd.DataFrame(component_rows).sort_values(
    ["split_seed", "target", "PATNO"]
).reset_index(drop=True)
selected_pipelines = pd.DataFrame(selected_rows).sort_values(
    ["split_seed", "target"]
).reset_index(drop=True)
print(f"Outer evaluation complete: {len(outer_predictions):,} paired predictions.")

Resuming with 0/20 outer seeds complete.
Completed 1/20 outer seeds; seed 00 took 0.0 min; estimated remaining 0.4 min
Completed 2/20 outer seeds; seed 01 took 0.0 min; estimated remaining 0.3 min
Completed 3/20 outer seeds; seed 02 took 0.0 min; estimated remaining 0.3 min
Completed 4/20 outer seeds; seed 03 took 0.0 min; estimated remaining 0.3 min
Completed 5/20 outer seeds; seed 04 took 0.0 min; estimated remaining 0.2 min
Completed 6/20 outer seeds; seed 05 took 0.0 min; estimated remaining 0.2 min
Completed 7/20 outer seeds; seed 06 took 0.0 min; estimated remaining 0.2 min
Completed 8/20 outer seeds; seed 07 took 0.0 min; estimated remaining 0.2 min
Completed 9/20 outer seeds; seed 08 took 0.0 min; estimated remaining 0.2 min
Completed 10/20 outer seeds; seed 09 took 0.0 min; estimated remaining 0.1 min
Completed 11/20 outer seeds; seed 10 took 0.0 min; estimated remaining 0.1 min
Completed 12/20 outer seeds; seed 11 took 0.0 min; estimated remaining 0.1 min
Completed 13/20 oute

## 20. Calculate seed-level performance

Report macro F1, all three recalls, balanced accuracy, accuracy, and macro one-vs-rest AUC for direct and hard-routed two-stage predictions.

In [20]:
def multiclass_metrics(truth, predictions, scores):
    recalls = recall_score(
        truth,
        predictions,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    return {
        "macro_f1": f1_score(
            truth,
            predictions,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            truth,
            predictions,
            labels=[0, 1, 2],
            average="weighted",
            zero_division=0,
        ),
        "accuracy": accuracy_score(truth, predictions),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "macro_auc_ovr": roc_auc_score(
            truth,
            scores,
            labels=[0, 1, 2],
            multi_class="ovr",
            average="macro",
        ),
        "recall_no_fall": recalls[0],
        "recall_rare_fall": recalls[1],
        "recall_recurrent_fall": recalls[2],
    }


metric_rows = []
confusion_rows = []
for (split_seed, system), group in outer_predictions.groupby(
    ["split_seed", "system"],
    sort=True,
):
    truth = group["true_class"].to_numpy()
    predicted = group["predicted_class"].to_numpy()
    scores = group[[
        "score_no_fall", "score_rare_fall", "score_recurrent_fall",
    ]].to_numpy()
    metric_rows.append({
        "split_seed": int(split_seed),
        "system": system,
        **multiclass_metrics(truth, predicted, scores),
    })
    matrix = confusion_matrix(truth, predicted, labels=[0, 1, 2])
    for true_class in range(3):
        for predicted_class in range(3):
            confusion_rows.append({
                "split_seed": int(split_seed),
                "system": system,
                "true_class": true_class,
                "predicted_class": predicted_class,
                "patients": int(matrix[true_class, predicted_class]),
            })

outer_metrics = pd.DataFrame(metric_rows)
outer_confusions = pd.DataFrame(confusion_rows)

component_metric_rows = []
for (split_seed, target_name), group in component_predictions.groupby(
    ["split_seed", "target"],
    sort=True,
):
    if target_name == "stage_2":
        group = group.loc[group["eligible_for_component_metric"].astype(bool)]
    truth = group["true_class"].to_numpy()
    predicted = group["predicted_class"].to_numpy()
    recalls = recall_score(
        truth,
        predicted,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )
    component_metric_rows.append({
        "split_seed": int(split_seed),
        "target": target_name,
        "patients": len(group),
        "macro_f1": f1_score(
            truth,
            predicted,
            labels=[0, 1],
            average="macro",
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(truth, predicted),
        "accuracy": accuracy_score(truth, predicted),
        "recall_class_0": recalls[0],
        "recall_class_1": recalls[1],
        "priority_recall": recalls[0] if target_name == "stage_2" else recalls[1],
    })
component_metrics = pd.DataFrame(component_metric_rows)

display(outer_metrics.groupby("system")[[
    "macro_f1", "balanced_accuracy", "accuracy",
    "recall_no_fall", "recall_rare_fall", "recall_recurrent_fall",
]].mean().round(3))
display(component_metrics.groupby("target")[[
    "macro_f1", "balanced_accuracy", "priority_recall",
]].mean().round(3))

,macro_f1,balanced_accuracy,accuracy,recall_no_fall,recall_rare_fall,recall_recurrent_fall
system,,,,,,
direct,0.513,0.550,0.624,0.710,0.384,0.557
two_stage,0.510,0.538,0.636,0.743,0.351,0.522


,macro_f1,balanced_accuracy,priority_recall
target,,,
stage_1,0.675,0.685,0.628
stage_2,0.665,0.670,0.803


## 21. Summarize paired performance and uncertainty

Average the 20 seed-level estimates and compare direct with two-stage on the same test patients. The corrected interval is exploratory because the repeated holdouts overlap.

In [21]:
summary_metrics = [
    "macro_f1", "weighted_f1", "accuracy", "balanced_accuracy", "macro_auc_ovr",
    "recall_no_fall", "recall_rare_fall", "recall_recurrent_fall",
]
system_summary = outer_metrics.groupby("system", as_index=False).agg(
    **{f"mean_{metric}": (metric, "mean") for metric in summary_metrics},
    **{f"sd_{metric}": (metric, "std") for metric in summary_metrics},
)

paired = outer_metrics.pivot(
    index="split_seed",
    columns="system",
    values=[
        "macro_f1", "balanced_accuracy", "accuracy",
        "recall_no_fall", "recall_rare_fall", "recall_recurrent_fall",
    ],
)
paired_differences = pd.DataFrame({"split_seed": paired.index.to_numpy()})
for metric in [
    "macro_f1", "balanced_accuracy", "accuracy",
    "recall_no_fall", "recall_rare_fall", "recall_recurrent_fall",
]:
    paired_differences[f"direct_minus_two_stage_{metric}"] = (
        paired[(metric, "direct")] - paired[(metric, "two_stage")]
    ).to_numpy()

correction = (1 / len(split_seeds)) + (312 / 728)
uncertainty_rows = []
for column in paired_differences.columns:
    if not column.startswith("direct_minus_two_stage_"):
        continue
    differences = paired_differences[column].to_numpy()
    mean_difference = float(differences.mean())
    standard_error = float(differences.std(ddof=1) * np.sqrt(correction))
    if standard_error > 0:
        low, high = stats.t.interval(
            0.95,
            len(differences) - 1,
            loc=mean_difference,
            scale=standard_error,
        )
        p_value = float(2 * stats.t.sf(
            abs(mean_difference / standard_error),
            df=len(differences) - 1,
        ))
    else:
        low, high, p_value = np.nan, np.nan, np.nan
    uncertainty_rows.append({
        "metric": column.removeprefix("direct_minus_two_stage_"),
        "mean_direct_minus_two_stage": mean_difference,
        "ci95_low": low,
        "ci95_high": high,
        "corrected_resampled_p_value": p_value,
        "n_outer_seeds": len(differences),
        "test_train_ratio": 312 / 728,
    })
paired_uncertainty = pd.DataFrame(uncertainty_rows)

display(system_summary.round(4))
display(paired_uncertainty.round(4))

,system,mean_macro_f1,mean_weighted_f1,mean_accuracy,mean_balanced_accuracy,mean_macro_auc_ovr,mean_recall_no_fall,mean_recall_rare_fall,mean_recall_recurrent_fall,sd_macro_f1,sd_weighted_f1,sd_accuracy,sd_balanced_accuracy,sd_macro_auc_ovr,sd_recall_no_fall,sd_recall_rare_fall,sd_recall_recurrent_fall
0,direct,0.5132,0.6416,0.6240,0.5501,0.7525,0.7098,0.3838,0.5567,0.0194,0.0155,0.0220,0.0305,0.0203,0.0449,0.0560,0.1015
1,two_stage,0.5098,0.6456,0.6359,0.5383,0.7475,0.7425,0.3507,0.5217,0.0189,0.0155,0.0201,0.0358,0.0296,0.0377,0.0684,0.1587


,metric,mean_direct_minus_two_stage,ci95_low,ci95_high,corrected_resampled_p_value,n_outer_seeds,test_train_ratio
0,macro_f1,0.0034,-0.0338,0.0406,0.8502,20,0.4286
1,balanced_accuracy,0.0118,-0.0450,0.0686,0.6687,20,0.4286
2,accuracy,-0.0119,-0.0385,0.0147,0.3625,20,0.4286
3,recall_no_fall,-0.0327,-0.0896,0.0242,0.2434,20,0.4286
4,recall_rare_fall,0.0331,-0.0529,0.1191,0.4305,20,0.4286
5,recall_recurrent_fall,0.0350,-0.1712,0.2412,0.7264,20,0.4286


## 22. Summarize selected models and features

Count the locally selected model families and transformed columns across seeds. These frequencies describe stability; they do not create a new global subset.

In [22]:
outer_winner_frequency = (
    selected_pipelines.groupby(["target", "model_family"])
    .size()
    .rename("selected_seeds")
    .reset_index()
    .sort_values(["target", "selected_seeds"], ascending=[True, False])
)
selected_column_frequency = (
    selected_pipelines.assign(
        selected_column=selected_pipelines["selected_column_names"].str.split(
            " | ",
            regex=False,
        )
    )
    .explode("selected_column")
    .groupby(["target", "selected_column"], as_index=False)
    .size()
    .rename(columns={"size": "selected_seeds"})
    .sort_values(["target", "selected_seeds"], ascending=[True, False])
)
confusion_summary = (
    outer_confusions.groupby(
        ["system", "true_class", "predicted_class"],
        as_index=False,
    )["patients"].sum()
)

display(outer_winner_frequency)
display(selected_column_frequency.groupby("target").head(10))

,target,model_family,selected_seeds
3,direct,random_forest,9
0,direct,catboost,7
4,direct,rbf_svc,2
1,direct,extra_trees,1
2,direct,linear_svc,1
6,stage_1,extra_trees,6
9,stage_1,rbf_svc,5
7,stage_1,logistic,4
8,stage_1,random_forest,3
5,stage_1,catboost,2


,target,selected_column,selected_seeds
28,direct,FOG_FORM_MISSING,20
29,direct,FRZGT12M,20
35,direct,NHY_COMBINED_MAX,20
40,direct,NP3GAIT_COMBINED_MAX,20
41,direct,NP3PSTBL_COMBINED_MAX,20
43,direct,NP4TOT,20
44,direct,NQ_FORM_MISSING,20
45,direct,NQ_GAUSSIAN_REVISION,20
46,direct,PART_IV_FORM_MISSING,20
54,direct,Years_since_PD_diagnosis,20


## 23. Validate the complete notebook

Confirm tuning integrity, paired outer coverage, valid scores, three refitted winners per seed, and complete reporting artifacts.

In [23]:
score_columns = [
    "score_no_fall", "score_rare_fall", "score_recurrent_fall",
]
outer_validation = pd.DataFrame([
    {
        "check": "focused tuning validation passed",
        "passed": tuning_validation["passed"].all(),
        "detail": f"{len(tuning_validation)} tuning checks",
    },
    {
        "check": "all 20 outer seeds completed",
        "passed": completed_outer_seeds == set(split_seeds),
        "detail": "one checkpoint per seed",
    },
    {
        "check": "direct and two-stage each have 20 seed metrics",
        "passed": set(outer_metrics["system"]) == {"direct", "two_stage"}
        and outer_metrics.groupby("system").size().eq(20).all(),
        "detail": "paired systems",
    },
    {
        "check": "each system predicts 312 patients per seed",
        "passed": outer_predictions.groupby(
            ["split_seed", "system"]
        )["PATNO"].nunique().eq(312).all(),
        "detail": "untouched 30% holdout",
    },
    {
        "check": "one prediction per patient, seed, and system",
        "passed": outer_predictions.groupby(
            ["split_seed", "system", "PATNO"]
        ).size().eq(1).all(),
        "detail": "no duplicate test predictions",
    },
    {
        "check": "systems use identical patient rows",
        "passed": outer_predictions.groupby(
            ["split_seed", "PATNO"]
        )["system"].nunique().eq(2).all(),
        "detail": "paired comparison",
    },
    {
        "check": "all scores are finite and sum to one",
        "passed": np.isfinite(outer_predictions[score_columns]).all().all()
        and np.allclose(outer_predictions[score_columns].sum(axis=1), 1.0),
        "detail": "predict_proba or fixed decision-score conversion",
    },
    {
        "check": "all outcome classes occur in every test set",
        "passed": outer_predictions.loc[
            outer_predictions["system"].eq("direct")
        ].groupby("split_seed")["true_class"].nunique().eq(3).all(),
        "detail": "stratified splits",
    },
    {
        "check": "three frozen winners refitted per seed",
        "passed": selected_pipelines.groupby(
            "split_seed"
        )["target"].nunique().eq(3).all(),
        "detail": "direct, Stage 1, and Stage 2",
    },
    {
        "check": "Stage 2 trains only on true fallers",
        "passed": selected_pipelines.loc[
            selected_pipelines["target"].eq("stage_2"),
            "training_patients",
        ].lt(728).all(),
        "detail": "fewer patients than direct and Stage 1",
    },
    {
        "check": "all refitted pipelines retained columns",
        "passed": selected_pipelines["selected_column_count"].gt(0).all(),
        "detail": "no empty final model input",
    },
    {
        "check": "outer metrics are finite and bounded",
        "passed": outer_metrics[summary_metrics].apply(
            lambda column: np.isfinite(column).all() and column.between(0, 1).all()
        ).all(),
        "detail": "20 paired seed estimates",
    },
    {
        "check": "run identity still matches",
        "passed": json.loads(manifest_path.read_text())["configuration_hash"]
        == configuration_hash,
        "detail": "same frozen inputs and grids",
    },
])
display(outer_validation)
assert outer_validation["passed"].all(), outer_validation.loc[
    ~outer_validation["passed"]
]
print(
    f"Complete validation passed: {outer_validation['passed'].sum()}/"
    f"{len(outer_validation)}"
)

,check,passed,detail
0,focused tuning validation passed,True,9 tuning checks
1,all 20 outer seeds completed,True,one checkpoint per seed
2,direct and two-stage each have 20 seed metrics,True,paired systems
3,each system predicts 312 patients per seed,True,untouched 30% holdout
4,"one prediction per patient, seed, and system",True,no duplicate test predictions
5,systems use identical patient rows,True,paired comparison
6,all scores are finite and sum to one,True,predict_proba or fixed decision-score conversion
7,all outcome classes occur in every test set,True,stratified splits
8,three frozen winners refitted per seed,True,"direct, Stage 1, and Stage 2"
9,Stage 2 trains only on true fallers,True,fewer patients than direct and Stage 1


Complete validation passed: 13/13


## 24. Save the final evaluation artifacts

Save patient-level predictions, seed-level metrics, paired comparisons, selection frequencies, and validation tables. Existing differing artifacts are never overwritten.

In [24]:
final_artifacts = {
    "outer_predictions.csv": outer_predictions,
    "component_predictions.csv": component_predictions,
    "outer_seed_metrics.csv": outer_metrics,
    "component_outer_seed_metrics.csv": component_metrics,
    "system_summary.csv": system_summary,
    "outer_confusion_matrices.csv": outer_confusions,
    "confusion_summary.csv": confusion_summary,
    "paired_architecture_differences.csv": paired_differences,
    "paired_corrected_uncertainty.csv": paired_uncertainty,
    "selected_pipelines.csv": selected_pipelines,
    "selected_model_family_frequency.csv": outer_winner_frequency,
    "selected_column_frequency.csv": selected_column_frequency,
    "outer_evaluation_validation.csv": outer_validation,
}
final_save_rows = []
for filename, frame in final_artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    final_save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

display(pd.DataFrame(final_save_rows))
print(
    "Focused tuning and one-time outer evaluation are complete. "
    "Outer results are reporting evidence and cannot revise the selected procedure."
)

,artifact,status,rows
0,outer_predictions.csv,created,12480
1,component_predictions.csv,created,12480
2,outer_seed_metrics.csv,created,40
3,component_outer_seed_metrics.csv,created,40
4,system_summary.csv,created,2
5,outer_confusion_matrices.csv,created,360
6,confusion_summary.csv,created,18
7,paired_architecture_differences.csv,created,20
8,paired_corrected_uncertainty.csv,created,6
9,selected_pipelines.csv,created,60


Focused tuning and one-time outer evaluation are complete. Outer results are reporting evidence and cannot revise the selected procedure.


## Interpretation boundary

This notebook produces the final primary internal-validation estimates for the locked 1,040-patient, 26-feature workflow. Repeated holdouts overlap and the same cohort informed earlier exploration, so these results are not independent external validation. Sensitivity analyses must reuse the locked primary procedure and cannot replace it based on their test scores.